In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [2]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install wandb

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-g7mk9ug9/unsloth_ba8b05481b0d478dad68f7bfcfa8b9f1
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-g7mk9ug9/unsloth_ba8b05481b0d478dad68f7bfcfa8b9f1
  Resolved https://github.com/unslothai/unsloth.git to commit eb91752d4b80363d633d0bfee10dd2e24968f4f4
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [3]:
from datasets import load_dataset

# Load dataset from Hugging Face
dataset = load_dataset("b-mc2/sql-create-context", split="train")

# Define the prompt template
prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: {question}

### Input (Database Schema):
{context}

### Response (SQL Query):
{answer}"""

# Formatting function
def format_prompts(batch):
    texts = []
    for question, context, answer in zip(batch["question"], batch["context"], batch["answer"]):
        text = prompt_template.format(question=question, context=context, answer=answer)
        # Append the EOS token so the model knows when to stop generating
        texts.append(text + tokenizer.eos_token)
    return {"text": texts}

# Map the formatting function to the dataset
dataset = dataset.map(format_prompts, batched=True)

README.md:   0%|          | 0.00/4.43k [00:00<?, ?B/s]

sql_create_context_v4.json: reconstructing file:   0%|          |  0.00B / 21.8MB            

sql_create_context_v4.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/78577 [00:00<?, ? examples/s]

Map:   0%|          | 0/78577 [00:00<?, ? examples/s]

NameError: name 'tokenizer' is not defined

In [ ]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Good for text-to-SQL tasks

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit", # Pre-quantized for speed
    max_seq_length = max_seq_length,
    dtype = None, # Auto-detects float16 for T4
    load_in_4bit = True,
)

# Apply LoRA adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Rank
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
)

In [ ]:
 from trl import SFTTrainer
from transformers import TrainingArguments
import wandb

# Initialize W&B run
wandb.init(project="Text-to-SQL-LLM", name="llama-3-8b-qlora")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 200, # Start small to test, then increase for actual fine-tuning
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        report_to = "wandb", # Logs to Weights & Biases
    ),
)

# Start training
trainer.train()

In [ ]:
# Enable native, 2x faster inference
FastLanguageModel.for_inference(model)

test_prompt = prompt_template.format(
    question="How many users signed up in 2026?",
    context="CREATE TABLE users (id INT, signup_date DATE, name VARCHAR);",
    answer=""
)

inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

In [ ]:
# 1. Install Unsloth and required libraries
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install wandb

# 2. Login to Hugging Face and W&B using Kaggle Secrets
from kaggle_secrets import UserSecretsClient
import wandb
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

login(token=hf_token)
wandb.login(key=wandb_key)

In [ ]:
from datasets import load_dataset

# Load the SQL dataset
dataset = load_dataset("b-mc2/sql-create-context", split="train")

# Define how the LLM should see the data
prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: {question}

### Input (Database Schema):
{context}

### Response (SQL Query):
{answer}"""

# Apply the formatting to the entire dataset
def format_prompts(batch):
    texts = []
    for question, context, answer in zip(batch["question"], batch["context"], batch["answer"]):
        text = prompt_template.format(question=question, context=context, answer=answer)
        texts.append(text) 
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)

# Print one example to verify it looks correct
print(dataset[0]["text"])

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import wandb

# Initialize Weights & Biases
wandb.init(project="Text-to-SQL-LLM", name="llama-3-8b-qlora-run1")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60, # Small number of steps for testing
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        report_to = "wandb", # Sends training graphs to W&B
    ),
)

# Execute Fine-Tuning
trainer.train()
wandb.finish()

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

# 1. Load the Model & Tokenizer FIRST
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. Add LoRA Adapters (This makes fine-tuning possible on a T4 GPU)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
)

# 3. Load and Format Dataset NOW (since the tokenizer is ready)
dataset = load_dataset("b-mc2/sql-create-context", split="train")

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: {question}

### Input (Database Schema):
{context}

### Response (SQL Query):
{answer}"""

def format_prompts(batch):
    texts = []
    for question, context, answer in zip(batch["question"], batch["context"], batch["answer"]):
        text = prompt_template.format(question=question, context=context, answer=answer)
        # Append the EOS token so the model knows when to stop generating text
        texts.append(text + tokenizer.eos_token)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print("\n✅ Dataset formatted! First example:")
print(dataset[0]["text"])

In [ ]:
# 1. Login to Hugging Face and W&B
from kaggle_secrets import UserSecretsClient
import wandb
from huggingface_hub import login

try:
    user_secrets = UserSecretsClient()
    hf_token = user_secrets.get_secret("HF_TOKEN")
    wandb_key = user_secrets.get_secret("WANDB_API_KEY")
    
    print("✅ Secrets loaded successfully!")
    
    # Log in using the loaded keys
    login(token=hf_token)
    wandb.login(key=wandb_key)
    print("✅ Successfully logged into Hugging Face and W&B!")
    
except Exception as e:
    print("❌ ERROR: Could not find secrets. Make sure the checkboxes in Add-ons -> Secrets are ticked!")
    print(e)

In [ ]:
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

# 1. Load the Model & Tokenizer
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. Add LoRA Adapters (This makes fine-tuning possible on a single T4 GPU)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
)

# 3. Load and Format Dataset 
dataset = load_dataset("b-mc2/sql-create-context", split="train")

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: {question}

### Input (Database Schema):
{context}

### Response (SQL Query):
{answer}"""

def format_prompts(batch):
    texts = []
    for question, context, answer in zip(batch["question"], batch["context"], batch["answer"]):
        text = prompt_template.format(question=question, context=context, answer=answer)
        # Append the EOS token so the model knows when to stop generating text
        texts.append(text + tokenizer.eos_token)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print("\n✅ Dataset formatted! First example:")
print(dataset[0]["text"])

In [ ]:
# 1. Login to Hugging Face and W&B
from kaggle_secrets import UserSecretsClient
import wandb
from huggingface_hub import login

user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")
wandb_key = user_secrets.get_secret("WANDB_API_KEY")

login(token=hf_token)
wandb.login(key=wandb_key)
print("✅ Successfully logged into Hugging Face and W&B!")

In [ ]:
import unsloth
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

# 1. Load the Model & Tokenizer
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. Add LoRA Adapters
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
)

# 3. Load and Format Dataset 
dataset = load_dataset("b-mc2/sql-create-context", split="train")

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: {question}

### Input (Database Schema):
{context}

### Response (SQL Query):
{answer}"""

def format_prompts(batch):
    texts = []
    for question, context, answer in zip(batch["question"], batch["context"], batch["answer"]):
        text = prompt_template.format(question=question, context=context, answer=answer)
        texts.append(text + tokenizer.eos_token)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print("\n✅ Dataset formatted! First example:")
print(dataset[0]["text"])

In [ ]:
!pip install "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install wandb

In [4]:
!pip install --upgrade huggingface_hub transformers

  Using cached transformers-5.17.0-py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 52.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 91.0 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.6/69.6 kB 4.2 MB/s eta 0:00:00
  Attempting uninstall: idna
    Found existing installation: idna 3.13
    Uninstalling idna-3.13:
      Successfully uninstalled idna-3.13
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.5.0
    Uninstalling transformers-5.5.0:
      Successfully uninstalled transformers-5.5.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 

In [5]:
import unsloth
from unsloth import FastLanguageModel
import torch
from datasets import load_dataset

print("⏳ 1. Loading Model and Tokenizer (This takes 1-2 minutes)...")
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

print("⏳ 2. Applying LoRA adapters...")
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
)

print("⏳ 3. Downloading and formatting dataset...")
dataset = load_dataset("b-mc2/sql-create-context", split="train")

prompt_template = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: {question}

### Input (Database Schema):
{context}

### Response (SQL Query):
{answer}"""

def format_prompts(batch):
    texts = []
    for question, context, answer in zip(batch["question"], batch["context"], batch["answer"]):
        text = prompt_template.format(question=question, context=context, answer=answer)
        # Append the EOS token so the model knows when to stop generating
        texts.append(text + tokenizer.eos_token)
    return {"text": texts}

dataset = dataset.map(format_prompts, batched=True)
print("\n✅ SUCCESS! Dataset formatted! First example:")
print(dataset[0]["text"])

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
⏳ 1. Loading Model and Tokenizer (This takes 1-2 minutes)...
==((====))==  Unsloth 2026.9.12: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


total weights      : 5.208 GiB
no_split classes   : ['LlamaDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 10.191 GiB requested
  cuda:0  budget  12.95 GiB  weights  2.705 GiB  free 10.247 GiB  reserve 10.191 GiB
  cuda:1  budget  12.98 GiB  weights  2.502 GiB  free 10.476 GiB  reserve 10.033 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 14.391 -> 12.952 GiB, cuda:1 14.421 -> 12.979 GiB (memory the quantiser keeps for its own load-time buffers)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


⏳ 2. Applying LoRA adapters...


Unsloth 2026.9.12 patched 32 layers with 32 QKV layers, 32 O layers and 32 MLP layers.


⏳ 3. Downloading and formatting dataset...


Map:   0%|          | 0/78577 [00:00<?, ? examples/s]


✅ SUCCESS! Dataset formatted! First example:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: How many heads of the departments are older than 56 ?

### Input (Database Schema):
CREATE TABLE head (age INTEGER)

### Response (SQL Query):
SELECT COUNT(*) FROM head WHERE age > 56<|eot_id|>


In [8]:
from trl import SFTTrainer
from transformers import TrainingArguments
import wandb

# Initialize Weights & Biases
wandb.init(project="Text-to-SQL-LLM", name="llama-3-8b-qlora-run1")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 60, # 60 steps for a quick test run
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        output_dir = "outputs",
        optim = "adamw_8bit",
        report_to = "wandb", 
    ),
)

# Execute Fine-Tuning
trainer.train()
wandb.finish()

train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,▁▂▂▂▂▃█▂▂▂▂▄▂▂▃▄▅▃▃▃▃▂▆▂▂▃▃▃▂▃▃▃▃▄▃▄▃▃▃▄
train/learning_rate,▁▂▂▃▄▅▆▇▇██▇▇▇▇▇▇▆▆▆▆▆▆▅▅▅▅▅▅▄▄▄▄▄▄▃▃▃▃▃
train/loss,▁▆▅▃▇▆█▂▄▄▅▄▃▁▂▅▅▅▃▄▄▆▃▅▃▄▂▇▃▅▆▄▅▃▅▄▅▂▅▃
train/epoch,0.00509
train/global_step,50
train/grad_norm,0.61652
train/learning_rate,4e-05
train/loss,0.57982


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 78,577 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 41,943,040 of 8,072,204,288 (0.52% trained)


Step,Training Loss
1,0.412515
2,0.546167
3,0.476934
4,0.417042
5,0.540213
6,0.479336
7,0.503910
8,0.554838
9,0.397145
10,0.465609


train/epoch,▁▁▁▁▁▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇██
train/global_step,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇█████
train/grad_norm,▁▂▂▃▃▆▄▅▅▅▄▆▅▇▆▇▆▇▆█▆▆▆▇▅▇▅▆▅▆▇▆▆▇█▇█▇▇▅
train/learning_rate,▁▂▂▃▄▅▆▇███▇▇▇▇▆▆▆▆▆▅▅▅▅▅▄▄▄▄▄▃▃▃▂▂▂▂▂▁▁
train/loss,▁▄▁▄▃▄▁▂▂▂▁▃▂▂▃▁▃▃▃▂▃▄▃▄▄▃▅▄▅▅▆▅▅▄█▆▆█▃▅
total_flos,2527053559627776.0
train/epoch,0.00611
train/global_step,60
train/grad_norm,0.86857
train/learning_rate,0.0
train/loss,0.56336


In [9]:
# 1. Put the model into high-speed inference mode
FastLanguageModel.for_inference(model)

# 2. Give it a brand new question and database schema
test_prompt = prompt_template.format(
    question="What is the total revenue for the electronics department in 2026?",
    context="CREATE TABLE sales (department VARCHAR, revenue DECIMAL, sale_year INTEGER)",
    answer="" # Leave this blank, the AI will fill it in!
)

# 3. Translate the text into numbers for the AI
inputs = tokenizer([test_prompt], return_tensors = "pt").to("cuda")

# 4. Generate the SQL!
outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)

# 5. Print the result
print("\n🤖 AI GENERATED SQL:")
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

Both `max_new_tokens` (=64) and `max_length`(=8192) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



🤖 AI GENERATED SQL:
Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
Write a SQL query to answer the following question: What is the total revenue for the electronics department in 2026?

### Input (Database Schema):
CREATE TABLE sales (department VARCHAR, revenue DECIMAL, sale_year INTEGER)

### Response (SQL Query):
SELECT SUM(revenue) FROM sales WHERE department = "Electronics" AND sale_year = 2026


In [10]:
import pandas as pd
from tqdm import tqdm

# 1. Grab 100 examples from the end of the dataset (unseen data)
test_data = dataset.select(range(len(dataset)-100, len(dataset)))
FastLanguageModel.for_inference(model)

correct_matches = 0
total_tested = len(test_data)
results = []

print(f"Starting evaluation on {total_tested} test queries...\n")

for i in tqdm(range(total_tested)):
    question = test_data[i]["question"]
    context = test_data[i]["context"]
    true_sql = test_data[i]["answer"]
    
    # Format the prompt WITHOUT the answer
    test_prompt = prompt_template.format(
        question=question, 
        context=context, 
        answer=""
    )
    
    # Generate the prediction
    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
    prompt_length = inputs.input_ids.shape[1]
    
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    # Extract only the newly generated text (ignoring the prompt)
    generated_tokens = outputs[0][prompt_length:]
    generated_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    # Normalize strings (lowercase and remove extra spaces) for fair comparison
    pred_normalized = " ".join(generated_sql.lower().split())
    true_normalized = " ".join(true_sql.lower().split())
    
    # Check if Exact Match
    is_correct = (pred_normalized == true_normalized)
    if is_correct:
        correct_matches += 1
        
    results.append({
        "Question": question,
        "True SQL": true_sql,
        "Generated SQL": generated_sql,
        "Exact Match": is_correct
    })

# Calculate Final Metrics
accuracy = (correct_matches / total_tested) * 100
print(f"\n✅ Evaluation Complete!")
print(f"📊 Exact Match (EM) Accuracy: {accuracy:.2f}%")

# Save the results to a CSV so you can put it on your GitHub
df_results = pd.DataFrame(results)
df_results.to_csv("text_to_sql_evaluation_results.csv", index=False)
print("💾 Saved detailed results to 'text_to_sql_evaluation_results.csv'")

Starting evaluation on 100 test queries...



100%|██████████| 100/100 [02:45<00:00,  1.65s/it]


✅ Evaluation Complete!
📊 Exact Match (EM) Accuracy: 71.00%
💾 Saved detailed results to 'text_to_sql_evaluation_results.csv'


In [11]:
# 2. Add LoRA Adapters (UPDATED for higher accuracy)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,            # INCREASED from 16
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,   # INCREASED from 16
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
)

TypeError: Unsloth: Your model already has LoRA adapters. Your new parameters are different.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
import wandb

# Initialize Weights & Biases for Run 2
wandb.init(project="Text-to-SQL-LLM", name="llama-3-8b-qlora-1epoch")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 100,             # Increased to give the model time to warm up
        num_train_epochs = 1,           # ONE FULL EPOCH (replaces max_steps)
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",   # ADDED: Smooths out the learning curve
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs",
        optim = "adamw_8bit",
        report_to = "wandb", 
    ),
)

# Execute Fine-Tuning
trainer.train()
wandb.finish()

In [12]:
# 1. Reload the base model fresh
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. Apply the new, upgraded LoRA adapters (r = 32)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,            # Upgraded for higher capacity
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,   # Upgraded to match r
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
)

print("✅ Model successfully reset and upgraded with r=32 adapters!")

==((====))==  Unsloth 2026.9.12: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


total weights      : 5.208 GiB
no_split classes   : ['LlamaDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 7.334 GiB requested
  cuda:0  budget  10.00 GiB  weights  2.604 GiB  free  7.394 GiB  reserve  7.334 GiB
  cuda:1  budget  10.22 GiB  weights  2.604 GiB  free  7.617 GiB  reserve  7.275 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 11.108 -> 9.997 GiB, cuda:1 11.356 -> 10.221 GiB (memory the quantiser keeps for its own load-time buffers)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


✅ Model successfully reset and upgraded with r=32 adapters!


In [13]:
from unsloth import FastLanguageModel
import torch

# 1. Reload the base model fresh
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

# 2. Apply the upgraded LoRA adapters (r = 32)
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,            # Upgraded for higher capacity
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,   # Upgraded to match r
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
)

print("✅ Model successfully reset and upgraded with r=32 adapters!")

==((====))==  Unsloth 2026.9.12: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


total weights      : 5.208 GiB
no_split classes   : ['LlamaDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 4.931 GiB requested
  cuda:0  budget   7.60 GiB  weights  2.604 GiB  free  4.992 GiB  reserve  4.931 GiB
  cuda:1  budget   7.82 GiB  weights  2.604 GiB  free  5.212 GiB  reserve  4.870 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 8.440 -> 7.596 GiB, cuda:1 8.684 -> 7.816 GiB (memory the quantiser keeps for its own load-time buffers)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


✅ Model successfully reset and upgraded with r=32 adapters!


In [14]:
from trl import SFTTrainer
from transformers import TrainingArguments
import wandb

# Initialize Weights & Biases for the full training run
wandb.init(project="Text-to-SQL-LLM", name="llama-3-8b-qlora-1epoch")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 100,             
        num_train_epochs = 1,           # Trains on the entire dataset once
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",   # Smooths out the learning curve
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs",
        optim = "adamw_8bit",
        report_to = "wandb", 
    ),
)

# Execute Full Fine-Tuning
trainer.train()
wandb.finish()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 78,577 | Num Epochs = 1 | Total steps = 9,823
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss
10,2.910335
20,2.082682


Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.
Unsloth: Will smartly offload gradients to save VRAM!


KeyboardInterrupt: 

In [ ]:
import pandas as pd
from tqdm import tqdm

# 1. Grab 100 test examples from the dataset that the model hasn't seen
test_data = dataset.select(range(len(dataset)-100, len(dataset)))
FastLanguageModel.for_inference(model)

correct_matches = 0
total_tested = len(test_data)
results = []

print(f"Starting evaluation on {total_tested} test queries...\n")

for i in tqdm(range(total_tested)):
    question = test_data[i]["question"]
    context = test_data[i]["context"]
    true_sql = test_data[i]["answer"]
    
    # Format prompt without the answer
    test_prompt = prompt_template.format(
        question=question, 
        context=context, 
        answer=""
    )
    
    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
    prompt_length = inputs.input_ids.shape[1]
    
    # Generate SQL prediction
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    generated_tokens = outputs[0][prompt_length:]
    generated_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    # Normalize strings for fair comparison
    pred_normalized = " ".join(generated_sql.lower().split())
    true_normalized = " ".join(true_sql.lower().split())
    
    is_correct = (pred_normalized == true_normalized)
    if is_correct:
        correct_matches += 1
        
    results.append({
        "Question": question,
        "True SQL": true_sql,
        "Generated SQL": generated_sql,
        "Exact Match": is_correct
    })

# Compute final percentage
accuracy = (correct_matches / total_tested) * 100
print(f"\n✅ Evaluation Complete!")
print(f"📊 Exact Match (EM) Accuracy: {accuracy:.2f}%")

In [16]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,             
        max_steps = 1000,               # CHANGED: 1,000 steps is plenty for a killer portfolio project
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",   
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs",
        optim = "adamw_8bit",
        report_to = "wandb", 
    ),
)

In [17]:
from unsloth import FastLanguageModel
import torch
from trl import SFTTrainer
from transformers import TrainingArguments
import wandb

print("⏳ 1. Resetting model with r=32 adapters...")
max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,            
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 32,   
    lora_dropout = 0, 
    bias = "none",
    use_gradient_checkpointing = "unsloth", 
)

print("🚀 2. Starting optimized 1,000-step training run...")
wandb.init(project="Text-to-SQL-LLM", name="llama-3-8b-qlora-1000steps")

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset, # Uses the dataset we formatted earlier
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 50,             
        max_steps = 1000,               # Optimized sweet spot (~1.5 hours)
        learning_rate = 2e-4,
        lr_scheduler_type = "cosine",   
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 10,
        output_dir = "outputs",
        optim = "adamw_8bit",
        report_to = "wandb", 
    ),
)

trainer.train()
wandb.finish()
print("✅ Training complete!")

⏳ 1. Resetting model with r=32 adapters...
==((====))==  Unsloth 2026.9.12: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


total weights      : 5.208 GiB
no_split classes   : ['LlamaDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 4.496 GiB requested
  cuda:0  budget   7.19 GiB  weights  2.604 GiB  free  4.585 GiB  reserve  4.496 GiB
  cuda:1  budget   7.35 GiB  weights  2.604 GiB  free  4.748 GiB  reserve  4.406 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 7.987 -> 7.188 GiB, cuda:1 8.169 -> 7.352 GiB (memory the quantiser keeps for its own load-time buffers)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 2
   \\   /|    Num examples = 78,577 | Num Epochs = 1 | Total steps = 1,000
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 83,886,080 of 8,114,147,328 (1.03% trained)


Step,Training Loss
10,2.784656
20,1.489781
30,0.851584
40,0.734659
50,0.679959
60,0.664185
70,0.699661
80,0.620076
90,0.657333
100,0.611430


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-1000/tokenizer_config.json.


train/epoch,▁▁▁▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇██
train/global_step,▁▁▁▁▁▂▂▂▂▂▂▂▃▃▃▃▄▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇███
train/grad_norm,█▅▇▃▂▅▁▂▂▁▂▂▁▃▁▂▃▁▂▁▃▂▁▂▁▁▁▁▂▁▁▁▂▂▁▃▂▁▁▂
train/learning_rate,▂▄▅▆██████▇▇▇▇▆▆▆▅▅▅▄▄▄▃▃▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁
train/loss,█▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
total_flos,4.271774963068109e+16
train/epoch,0.10181
train/global_step,1000
train/grad_norm,0.30622
train/learning_rate,0.0
train/loss,0.50543


✅ Training complete!


In [18]:
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel

# 1. Grab 100 test examples from the end of the dataset (unseen data)
test_data = dataset.select(range(len(dataset)-100, len(dataset)))
FastLanguageModel.for_inference(model)

correct_matches = 0
total_tested = len(test_data)
results = []

print(f"Starting evaluation on {total_tested} test queries...\n")

for i in tqdm(range(total_tested)):
    question = test_data[i]["question"]
    context = test_data[i]["context"]
    true_sql = test_data[i]["answer"]
    
    # Format the prompt WITHOUT the answer
    test_prompt = prompt_template.format(
        question=question, 
        context=context, 
        answer=""
    )
    
    # Generate the prediction
    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
    prompt_length = inputs.input_ids.shape[1]
    
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    generated_tokens = outputs[0][prompt_length:]
    generated_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    # Normalize strings for fair comparison
    pred_normalized = " ".join(generated_sql.lower().split())
    true_normalized = " ".join(true_sql.lower().split())
    
    is_correct = (pred_normalized == true_normalized)
    if is_correct:
        correct_matches += 1
        
    results.append({
        "Question": question,
        "True SQL": true_sql,
        "Generated SQL": generated_sql,
        "Exact Match": is_correct
    })

accuracy = (correct_matches / total_tested) * 100
print(f"\n✅ Evaluation Complete!")
print(f"📊 New Exact Match (EM) Accuracy: {accuracy:.2f}%")

# Save the results to a CSV
df_results = pd.DataFrame(results)
df_results.to_csv("text_to_sql_evaluation_results.csv", index=False)
print("💾 Saved detailed results to 'text_to_sql_evaluation_results.csv'")

Starting evaluation on 100 test queries...



  0%|          | 0/100 [00:00<?, ?it/s]


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

In [19]:
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel

# 1. Grab 100 test examples from the end of the dataset (unseen data)
test_data = dataset.select(range(len(dataset)-100, len(dataset)))
FastLanguageModel.for_inference(model)

correct_matches = 0
total_tested = len(test_data)
results = []

print(f"Starting evaluation on {total_tested} test queries...\n")

for i in tqdm(range(total_tested)):
    question = test_data[i]["question"]
    context = test_data[i]["context"]
    true_sql = test_data[i]["answer"]
    
    # Format the prompt WITHOUT the answer
    test_prompt = prompt_template.format(
        question=question, 
        context=context, 
        answer=""
    )
    
    # Generate the prediction
    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
    prompt_length = inputs.input_ids.shape[1]
    
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    generated_tokens = outputs[0][prompt_length:]
    generated_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    # Normalize strings for fair comparison
    pred_normalized = " ".join(generated_sql.lower().split())
    true_normalized = " ".join(true_sql.lower().split())
    
    is_correct = (pred_normalized == true_normalized)
    if is_correct:
        correct_matches += 1
        
    results.append({
        "Question": question,
        "True SQL": true_sql,
        "Generated SQL": generated_sql,
        "Exact Match": is_correct
    })

accuracy = (correct_matches / total_tested) * 100
print(f"\n✅ Evaluation Complete!")
print(f"📊 Final Exact Match (EM) Accuracy: {accuracy:.2f}%")

# Save the results to a CSV
df_results = pd.DataFrame(results)
df_results.to_csv("text_to_sql_evaluation_results.csv", index=False)
print("💾 Saved detailed results to 'text_to_sql_evaluation_results.csv'")

Starting evaluation on 100 test queries...



  0%|          | 0/100 [00:00<?, ?it/s]


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

In [20]:
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel

# 1. Grab 100 test examples from the end of the dataset (unseen data)
test_data = dataset.select(range(len(dataset)-100, len(dataset)))
FastLanguageModel.for_inference(model)

correct_matches = 0
total_tested = len(test_data)
results = []

print(f"Starting evaluation on {total_tested} test queries...\n")

for i in tqdm(range(total_tested)):
    question = test_data[i]["question"]
    context = test_data[i]["context"]
    true_sql = test_data[i]["answer"]
    
    # Format the prompt WITHOUT the answer
    test_prompt = prompt_template.format(
        question=question, 
        context=context, 
        answer=""
    )
    
    # Explicitly send inputs to cuda:1 to match the multi-GPU setup
    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda:1")
    prompt_length = inputs.input_ids.shape[1]
    
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    generated_tokens = outputs[0][prompt_length:]
    generated_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    # Normalize strings for fair comparison
    pred_normalized = " ".join(generated_sql.lower().split())
    true_normalized = " ".join(true_sql.lower().split())
    
    is_correct = (pred_normalized == true_normalized)
    if is_correct:
        correct_matches += 1
        
    results.append({
        "Question": question,
        "True SQL": true_sql,
        "Generated SQL": generated_sql,
        "Exact Match": is_correct
    })

accuracy = (correct_matches / total_tested) * 100
print(f"\n✅ Evaluation Complete!")
print(f"📊 Final Exact Match (EM) Accuracy: {accuracy:.2f}%")

# Save the results to a CSV
df_results = pd.DataFrame(results)
df_results.to_csv("text_to_sql_evaluation_results.csv", index=False)
print("💾 Saved detailed results to 'text_to_sql_evaluation_results.csv'")

Starting evaluation on 100 test queries...



  0%|          | 0/100 [00:00<?, ?it/s]


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:1 and cuda:0!

In [21]:
import torch
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel

# 1. Force the model onto a single device (cuda:0) to clear multi-GPU splits
model.to("cuda:0")
FastLanguageModel.for_inference(model)

# 2. Grab 100 test examples from the end of the dataset (unseen data)
test_data = dataset.select(range(len(dataset)-100, len(dataset)))

correct_matches = 0
total_tested = len(test_data)
results = []

print(f"Starting evaluation on {total_tested} test queries...\n")

for i in tqdm(range(total_tested)):
    question = test_data[i]["question"]
    context = test_data[i]["context"]
    true_sql = test_data[i]["answer"]
    
    # Format the prompt WITHOUT the answer
    test_prompt = prompt_template.format(
        question=question, 
        context=context, 
        answer=""
    )
    
    # Send inputs to cuda:0 to match the single-device model
    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda:0")
    prompt_length = inputs.input_ids.shape[1]
    
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    generated_tokens = outputs[0][prompt_length:]
    generated_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    # Normalize strings for fair comparison
    pred_normalized = " ".join(generated_sql.lower().split())
    true_normalized = " ".join(true_sql.lower().split())
    
    is_correct = (pred_normalized == true_normalized)
    if is_correct:
        correct_matches += 1
        
    results.append({
        "Question": question,
        "True SQL": true_sql,
        "Generated SQL": generated_sql,
        "Exact Match": is_correct
    })

accuracy = (correct_matches / total_tested) * 100
print(f"\n✅ Evaluation Complete!")
print(f"📊 Final Exact Match (EM) Accuracy: {accuracy:.2f}%")

# Save the results to a CSV
df_results = pd.DataFrame(results)
df_results.to_csv("text_to_sql_evaluation_results.csv", index=False)
print("💾 Saved detailed results to 'text_to_sql_evaluation_results.csv'")

Starting evaluation on 100 test queries...



  0%|          | 0/100 [00:00<?, ?it/s]


RuntimeError: Expected all tensors to be on the same device, but got mat2 is on cuda:0, different from other tensors on cuda:1 (when checking argument in method wrapper_CUDA_mm)

In [22]:
import torch
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel

# 1. Clear GPU memory and reload the trained model cleanly onto a single device
torch.cuda.empty_cache()

max_seq_length = 2048
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "outputs",  # Loads your newly trained 1000-step checkpoint
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

FastLanguageModel.for_inference(model) # Optimizes model for fast generation

# 2. Grab 100 test examples from the end of the dataset (unseen data)
test_data = dataset.select(range(len(dataset)-100, len(dataset)))

correct_matches = 0
total_tested = len(test_data)
results = []

print(f"Starting evaluation on {total_tested} test queries...\n")

for i in tqdm(range(total_tested)):
    question = test_data[i]["question"]
    context = test_data[i]["context"]
    true_sql = test_data[i]["answer"]
    
    # Format the prompt WITHOUT the answer
    test_prompt = prompt_template.format(
        question=question, 
        context=context, 
        answer=""
    )
    
    # Send inputs to cuda:0
    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda")
    prompt_length = inputs.input_ids.shape[1]
    
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    generated_tokens = outputs[0][prompt_length:]
    generated_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    # Normalize strings for fair comparison
    pred_normalized = " ".join(generated_sql.lower().split())
    true_normalized = " ".join(true_sql.lower().split())
    
    is_correct = (pred_normalized == true_normalized)
    if is_correct:
        correct_matches += 1
        
    results.append({
        "Question": question,
        "True SQL": true_sql,
        "Generated SQL": generated_sql,
        "Exact Match": is_correct
    })

accuracy = (correct_matches / total_tested) * 100
print(f"\n✅ Evaluation Complete!")
print(f"📊 Final Exact Match (EM) Accuracy: {accuracy:.2f}%")

# Save the results to a CSV
df_results = pd.DataFrame(results)
df_results.to_csv("text_to_sql_evaluation_results.csv", index=False)
print("💾 Saved detailed results to 'text_to_sql_evaluation_results.csv'")

RuntimeError: Unsloth: Failed to load model. Both AutoConfig and PeftConfig loading failed.

AutoConfig error: Unrecognized model in outputs. Should have a `model_type` key in its config.json.

PeftConfig error: Can't find 'adapter_config.json' at 'outputs'



In [23]:
import torch
import pandas as pd
from tqdm import tqdm
from unsloth import FastLanguageModel
from peft import PeftModel

print("💾 1. Saving trained adapters locally...")
model.save_pretrained("sql_lora_adapter")
tokenizer.save_pretrained("sql_lora_adapter")

print("🧹 2. Clearing GPU memory...")
torch.cuda.empty_cache()

print("🔄 3. Reloading base model fresh on a single GPU (cuda:0)...")
max_seq_length = 2048
base_model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/llama-3-8b-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

print("🔗 4. Attaching your trained adapters...")
model = PeftModel.from_pretrained(base_model, "sql_lora_adapter")
FastLanguageModel.for_inference(model) # Optimizes for fast generation

# 5. Grab 100 test examples from the end of the dataset (unseen data)
test_data = dataset.select(range(len(dataset)-100, len(dataset)))

correct_matches = 0
total_tested = len(test_data)
results = []

print(f"\n🚀 Starting evaluation on {total_tested} test queries...\n")

for i in tqdm(range(total_tested)):
    question = test_data[i]["question"]
    context = test_data[i]["context"]
    true_sql = test_data[i]["answer"]
    
    # Format the prompt WITHOUT the answer
    test_prompt = prompt_template.format(
        question=question, 
        context=context, 
        answer=""
    )
    
    # Send inputs to cuda:0
    inputs = tokenizer([test_prompt], return_tensors="pt").to("cuda:0")
    prompt_length = inputs.input_ids.shape[1]
    
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    
    generated_tokens = outputs[0][prompt_length:]
    generated_sql = tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()
    
    # Normalize strings for fair comparison
    pred_normalized = " ".join(generated_sql.lower().split())
    true_normalized = " ".join(true_sql.lower().split())
    
    is_correct = (pred_normalized == true_normalized)
    if is_correct:
        correct_matches += 1
        
    results.append({
        "Question": question,
        "True SQL": true_sql,
        "Generated SQL": generated_sql,
        "Exact Match": is_correct
    })

accuracy = (correct_matches / total_tested) * 100
print(f"\n✅ Evaluation Complete!")
print(f"📊 Final Exact Match (EM) Accuracy: {accuracy:.2f}%")

# Save the results to a CSV
df_results = pd.DataFrame(results)
df_results.to_csv("text_to_sql_evaluation_results.csv", index=False)
print("💾 Saved detailed results to 'text_to_sql_evaluation_results.csv'")

💾 1. Saving trained adapters locally...


Unsloth: Restored added_tokens_decoder metadata in sql_lora_adapter/tokenizer_config.json.


🧹 2. Clearing GPU memory...
🔄 3. Reloading base model fresh on a single GPU (cuda:0)...
==((====))==  Unsloth 2026.9.12: Fast Llama patching. Transformers: 5.17.0.
   \\   /|    Tesla T4. Num GPUs = 2. Max memory: 14.562 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


/usr/local/lib/python3.12/dist-packages/huggingface_hub/constants.py:302: FutureWarning: The `HF_HUB_ENABLE_HF_TRANSFER` environment variable is deprecated as 'hf_transfer' is not used anymore. Please use `HF_XET_HIGH_PERFORMANCE` instead to enable high performance transfer with Xet. Visit https://huggingface.co/docs/huggingface_hub/package_reference/environment_variables#hfxethighperformance for more details.
  warnings.warn(


total weights      : 5.208 GiB
no_split classes   : ['LlamaDecoderLayer']
output head        : lm_head -> cuda:1
head headroom      : 0.342 GiB
activation reserve : 4.168 GiB requested
  cuda:0  budget   4.03 GiB  weights  1.486 GiB  free  2.545 GiB  reserve  2.519 GiB
  cuda:1  budget   9.85 GiB  weights  3.721 GiB  free  6.132 GiB  reserve  4.168 GiB   <- output head
note: Bnb4BitHfQuantizer.adjust_max_memory lowered the budget on cuda:0 4.479 -> 4.031 GiB, cuda:1 10.948 -> 9.853 GiB (memory the quantiser keeps for its own load-time buffers)


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/llama-3-8b-Instruct-bnb-4bit as a legacy tokenizer.


🔗 4. Attaching your trained adapters...

🚀 Starting evaluation on 100 test queries...



100%|██████████| 100/100 [02:54<00:00,  1.74s/it]


✅ Evaluation Complete!
📊 Final Exact Match (EM) Accuracy: 84.00%
💾 Saved detailed results to 'text_to_sql_evaluation_results.csv'
